# Battery SOH/RUL Prognostics — Merged Notebook

**Author:** Rajnish

This notebook consolidates Day 8 through Day 18 into a single, theme-organized notebook.
Each **Section** groups notebooks that address the same stage of the workflow, and each **Subsection** corresponds to one original day's notebook (with its own internal sub-steps preserved).

## Table of Contents
- Section 1: Data Preparation & Exploratory Analysis
  - Day 8 — Data Cleaning & Alignment
  - Day 9 — Temporal Shift & Feature Extraction
  - Day 10 — Temporal Exploratory Data Analysis (EDA)
- Section 2: Sequence Processing & Pipeline Construction
  - Day 11 — Sequence Processing
  - Day 12 — Sequence Pipeline (Train/Val/Test, Leakage-Safe)
- Section 3: Baseline Prognostics & Key Feature Analysis
  - Day 13 — Baseline Prognostic Models (SoH & RUL)
  - Day 14 — Key Change / Capacity Degradation Analysis
- Section 4: Deep Learning Models for SOH Prediction
  - Day 15 — LSTM Model for SOH Prediction
  - Day 16 — LSTM vs GRU Comparison
  - Day 17 — TCN (1D-CNN) Model for SOH Prediction
- Section 5: Final Model Comparison & Conclusion
  - Day 18 — Model Comparison & Discussion

# Section 1: Data Preparation & Exploratory Analysis

## Day 8: Data Cleaning & Alignment

*(Source notebook: `day8_Rajnish_Cleaning_alignment.ipynb`)*

### ***Importing Dataset***

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -o -q "/content/drive/MyDrive/cleaned_dataset.zip" -d /content/
print("done.")

In [ ]:
import os, glob
if os.path.exists("/content/cleaned_dataset.zip"):
    zip_path = "/content/cleaned_dataset.zip"
else:
    drive_zips = glob.glob("/content/drive/MyDrive/**/cleaned_dataset*.zip", recursive=True)
    if drive_zips:
        zip_path = drive_zips[0]
    else:
        raise FileNotFoundError("Zip file Not found")

print(f"Using zip: {zip_path}")
!unzip -o -q "{zip_path}" -d /content/

In [ ]:
import os, glob, re, numpy as np, pandas as pd

if 'df_meta' not in globals():
    meta_files = [os.path.join(r, f) for r, d, fs in os.walk('/content') for f in fs if f == 'metadata.csv' and 'drive' not in r]
    if not meta_files:
        drive_zips = glob.glob('/content/drive/MyDrive/**/cleaned_dataset*.zip', recursive=True)
        if drive_zips:
            !unzip -o -q "{drive_zips[0]}" -d /content/
        meta_files = [os.path.join(r, f) for r, d, fs in os.walk('/content') for f in fs if f == 'metadata.csv' and 'drive' not in r]

    METADATA_PATH = meta_files[0]
    df_meta = pd.read_csv(METADATA_PATH)

    def sanitize_cap(x):
        if pd.isna(x) or str(x).strip() in ['[]', '']:
            return np.nan
        try:
            v = float(x)
            return v if v > 0.05 else np.nan
        except:
            return np.nan

    df_meta['clean_capacity'] = df_meta['Capacity'].apply(sanitize_cap)

def generate_quality_table(df):
    records = []
    for b_id, grp in df.groupby('battery_id'):
        dis_grp = grp[grp['type'] == 'discharge']
        chg_grp = grp[grp['type'] == 'charge']
        imp_grp = grp[grp['type'] == 'impedance']
        valid_caps = dis_grp['clean_capacity'].dropna()
        temps = '/'.join(map(str, sorted(grp['ambient_temperature'].unique())))
        missing_count = len(dis_grp) - len(valid_caps)
        min_c = f"{valid_caps.min():.3f}" if len(valid_caps) > 0 else "N/A"
        max_c = f"{valid_caps.max():.3f}" if len(valid_caps) > 0 else "N/A"

        issues = []
        if missing_count > 0: issues.append(f"{missing_count} missing/aborted")
        if len(valid_caps) > 0 and valid_caps.max() > 2.2: issues.append("Cap > 2.2Ah spike")
        if len(valid_caps) > 0 and valid_caps.min() < 0.2: issues.append("Deep EOL (<0.2Ah)")

        records.append({
            'Battery_ID': b_id,
            'Ambient_Temp_C': temps,
            'Total_Cycles': len(grp),
            'Discharge_Count': len(dis_grp),
            'Charge_Count': len(chg_grp),
            'Impedance_Count': len(imp_grp),
            'Valid_Capacity_Cycles': len(valid_caps),
            'Missing_Cap_Count': missing_count,
            'Min_Capacity_Ah': min_c,
            'Max_Capacity_Ah': max_c,
            'Data_Quality_Notes': ', '.join(issues) if issues else 'Clean'
        })
    return pd.DataFrame(records)

df_quality = generate_quality_table(df_meta)
df_quality.to_csv('/content/battery_data_quality_report.csv', index=False)
display(df_quality)

In [ ]:
import os, glob, numpy as np, pandas as pd

if 'DATA_CSV_DIR' not in globals() or not os.path.exists(DATA_CSV_DIR):
    meta_files = [os.path.join(r, f) for r, d, fs in os.walk('/content') for f in fs if f == 'metadata.csv' and 'drive' not in r]
    if not meta_files:
        drive_zips = glob.glob('/content/drive/MyDrive/**/cleaned_dataset*.zip', recursive=True)
        if drive_zips:
            !unzip -o -q "{drive_zips[0]}" -d /content/
        meta_files = [os.path.join(r, f) for r, d, fs in os.walk('/content') for f in fs if f == 'metadata.csv' and 'drive' not in r]

    METADATA_PATH = meta_files[0]
    DATASET_DIR = os.path.dirname(METADATA_PATH)
    DATA_CSV_DIR = os.path.join(DATASET_DIR, 'data')
    df_meta = pd.read_csv(METADATA_PATH)
    df_meta['clean_capacity'] = pd.to_numeric(df_meta['Capacity'].astype(str).str.replace(r'[\[\]]', '', regex=True), errors='coerce')
    df_meta.loc[df_meta['clean_capacity'] <= 0.05, 'clean_capacity'] = np.nan

def extract_early_window_features(file_path, early_window_sec=500):
    if not os.path.exists(file_path):
        return None
    try:
        raw_df = pd.read_csv(file_path)
    except Exception:
        return None

    if len(raw_df) < 10 or 'Time' not in raw_df.columns or raw_df['Time'].iloc[-1] < 60:
        return None

    early_df = raw_df[raw_df['Time'] <= early_window_sec].copy()
    if len(early_df) < 5:
        return None

    v = early_df['Voltage_measured'].values
    i = early_df['Current_measured'].values
    t = early_df['Temperature_measured'].values
    time_pts = early_df['Time'].values

    dt_span = time_pts[-1] - time_pts[0] + 1e-6
    v_drop = v[0] - v[min(3, len(v)-1)]
    r_est = v_drop / (abs(i[min(3, len(i)-1)]) + 1e-6)

    return {
        'feat_R_est': r_est,
        'feat_V_drop_init': v_drop,
        'feat_dV_dt_early': (v[-1] - v[0]) / dt_span,
        'feat_dT_dt_early': (t[-1] - t[0]) / dt_span,
        'feat_V_mean_early': float(np.mean(v)),
        'feat_V_std_early': float(np.std(v)),
        'feat_T_mean_early': float(np.mean(t)),
        'feat_T_rise_early': float(t[-1] - t[0])
    }

valid_discharge_meta = df_meta[(df_meta['type'] == 'discharge') & (df_meta['clean_capacity'].notna())]
feature_rows = []

for idx, r in valid_discharge_meta.iterrows():
    fpath = os.path.join(DATA_CSV_DIR, r['filename'])
    feats = extract_early_window_features(fpath, early_window_sec=500)
    if feats is not None:
        feats['battery_id'] = r['battery_id']
        feats['test_id'] = int(r['test_id'])
        feats['ambient_temperature'] = r['ambient_temperature']
        feats['target_capacity'] = float(r['clean_capacity'])
        feature_rows.append(feats)

df_ml = pd.DataFrame(feature_rows)
df_ml.to_csv('/content/cleaned_ml_ready_battery_dataset.csv', index=False)
print("SHAPE:", df_ml.shape)
display(df_ml.head())

### ***Visual Rpresentation***

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold


gkf = GroupKFold(n_splits=4)
groups = df_ml['battery_id'].values
X = df_ml[[c for c in df_ml.columns if c.startswith('feat_')]]
y = df_ml['target_capacity'].values

print("4 FOLD GROUP BASED SPLIT (NO DATA LEAKAGE)")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    val_cells = sorted(list(set(groups[val_idx])))
    print(f"Fold {fold+1} Test Cells: {val_cells} (Samples: {len(val_idx)})")


plt.figure(figsize=(10, 5))
for cell in ['B0005', 'B0006', 'B0007', 'B0018']:
    sub = df_ml[df_ml['battery_id'] == cell]
    plt.plot(sub['test_id'], sub['target_capacity'], marker='.', label=f"Cell {cell}")

plt.xlabel("Cycle / Test ID")
plt.ylabel("Discharge Capacity (Ah)")
plt.title("Cleaned & Leak-Free Capacity Fade Curves")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## Day 9: Temporal Shift & Feature Extraction

*(Source notebook: `day9_Rajnish_Temposhift.ipynb`)*

### **Cell 1  Imports & Drive Mount**

In [ ]:
from google.colab import drive
import os, glob, json
import numpy as np, pandas as pd

drive.mount('/content/drive')

### ***Cell 2 Locate & Unzip Dataset***

In [ ]:
zip_files = glob.glob('/content/drive/MyDrive/**/cleaned_dataset*.zip', recursive=True)
if zip_files:
    os.system(f'unzip -o -q "{zip_files[0]}" -d /content/')

meta_path = next(
    os.path.join(r, f) for r, _, fs in os.walk('/content') for f in fs
    if f == 'metadata.csv' and 'drive' not in r
)
data_dir = os.path.join(os.path.dirname(meta_path), 'data')

print('metadata:', meta_path)
print('data_dir:', data_dir)

### ***Cell 3 Load Metadata & Compute SOH / RUL***

In [ ]:
df = pd.read_csv(meta_path)
df['Capacity'] = pd.to_numeric(
    df['Capacity'].astype(str).str.replace(r'[\[\]]', '', regex=True), errors='coerce'
)
dis_df = (df[(df['type'] == 'discharge') & (df['Capacity'] > 0.05)]
          .sort_values(['battery_id', 'test_id']).copy())

dis_df['SOH'] = dis_df['Capacity'] / dis_df.groupby('battery_id')['Capacity'].transform('first')

def add_rul(g):
    eol_mask = g['Capacity'] <= 0.7 * g['Capacity'].iloc[0]
    eol_test_id = g.loc[eol_mask, 'test_id'].iloc[0] if eol_mask.any() else g['test_id'].max()
    g['RUL'] = (eol_test_id - g['test_id']).clip(lower=0)
    return g

dis_df = dis_df.groupby('battery_id', group_keys=False).apply(add_rul)

print(dis_df.shape)
dis_df.head()

### ***Cell 4 Feature Extraction Function***

In [ ]:
def extract_features(row):
    f = os.path.join(data_dir, row['filename'])
    if not os.path.exists(f):
        return None
    try:
        raw = pd.read_csv(f)
        if len(raw) < 10 or 'Time' not in raw or raw['Time'].iloc[-1] < 60:
            return None
        w = raw[raw['Time'] <= 500]
        if len(w) < 5:
            return None
        i3 = min(3, len(w) - 1)
        dt = w['Time'].iloc[-1] - w['Time'].iloc[0] + 1e-6
        return {
            'battery_id': row['battery_id'], 'test_id': int(row['test_id']),
            'ambient_temperature': row['ambient_temperature'],
            'feat_R_est': (w['Voltage_measured'].iloc[0] - w['Voltage_measured'].iloc[i3])
                          / (abs(w['Current_measured'].iloc[i3]) + 1e-6),
            'feat_dV_dt': (w['Voltage_measured'].iloc[-1] - w['Voltage_measured'].iloc[0]) / dt,
            'feat_dT_dt': (w['Temperature_measured'].iloc[-1] - w['Temperature_measured'].iloc[0]) / dt,
            'feat_V_mean': float(w['Voltage_measured'].mean()),
            'feat_T_mean': float(w['Temperature_measured'].mean()),
            'target_capacity': row['Capacity'], 'target_SOH': row['SOH'], 'target_RUL': row['RUL']
        }
    except Exception:
        return None

### ***Cell 5 Run Extraction (Build data DataFrame)***

In [ ]:
records = [r for r in (extract_features(row) for _, row in dis_df.iterrows()) if r]
data = pd.DataFrame(records)

print(data.shape)
data.head()

### ***Cell 6 Split Functions***

In [ ]:
def cell_split(df, val_cells, test_cells):
    is_val, is_test = df['battery_id'].isin(val_cells), df['battery_id'].isin(test_cells)
    return df[~is_val & ~is_test], df[is_val], df[is_test]

def temporal_split(df, ratios=(0.6, 0.8)):
    parts = {'train': [], 'val': [], 'test': []}
    for _, g in df.groupby('battery_id'):
        n = len(g)
        parts['train'].append(g.iloc[:int(n*ratios[0])])
        parts['val'].append(g.iloc[int(n*ratios[0]):int(n*ratios[1])])
        parts['test'].append(g.iloc[int(n*ratios[1]):])
    return (pd.concat(parts['train']), pd.concat(parts['val']), pd.concat(parts['test']))

### ***Cell 7 Apply Splits***

In [ ]:
test_cells = ['B0018', 'B0028', 'B0032', 'B0036', 'B0040', 'B0043', 'B0044', 'B0051', 'B0056']
val_cells  = ['B0007', 'B0027', 'B0031', 'B0034', 'B0039', 'B0042', 'B0048', 'B0050', 'B0054']

cell_train, cell_val, cell_test = cell_split(data, val_cells, test_cells)
temp_train, temp_val, temp_test = temporal_split(data)

print('cell:', len(cell_train), len(cell_val), len(cell_test))
print('temporal:', len(temp_train), len(temp_val), len(temp_test))

### ***Cell 8 Save CSVs + Summary JSON***

In [ ]:
outputs = {
    'cell_train': cell_train, 'cell_val': cell_val, 'cell_test': cell_test,
    'temporal_train': temp_train, 'temporal_val': temp_val, 'temporal_test': temp_test
}
for name, d in outputs.items():
    d.to_csv(f'/content/{name}.csv', index=False)

summary = {
    'total_samples': len(data),
    'cells_count': int(data['battery_id'].nunique()),
    'cell_split': {k: len(outputs[f'cell_{k}']) for k in ['train', 'val', 'test']},
    'temporal_split': {k: len(outputs[f'temporal_{k}']) for k in ['train', 'val', 'test']}
}
with open('/content/data_prep_frozen_config.json', 'w') as fp:
    json.dump(summary, fp, indent=4)

print('SUCCESS:', json.dumps(summary, indent=2))

## Day 10: Temporal Exploratory Data Analysis (EDA)

*(Source notebook: `day10_Rajnish_TempEDA.ipynb`)*

### **Cell 1 Imports & Plotting Setup**

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

### **Cell 2 Dataset**

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/cleaned_dataset.zip"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError("cleaned_dataset.zip not found in My Drive.")

print("Dataset found:")
print(ZIP_PATH)

### ***Extract Dataset***

In [ ]:
import zipfile

EXTRACT_DIR = "/content/battery_dataset"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully.")

### ***Find Meta Data***

In [ ]:
import glob
import os

metadata_files = glob.glob(
    "/content/battery_dataset/**/metadata.csv",
    recursive=True
)

if not metadata_files:
    raise FileNotFoundError("metadata.csv not found inside the dataset.")

METADATA_PATH = metadata_files[0]

DATA_DIR = os.path.join(
    os.path.dirname(METADATA_PATH),
    "data"
)

print("Metadata:", METADATA_PATH)
print("Data directory:", DATA_DIR)

### **Cell 3 Load & Clean Data**

In [ ]:
df = pd.read_csv(METADATA_PATH)
df["Capacity"] = pd.to_numeric(
    df["Capacity"]
    .astype(str)
    .str.replace(r"[\[\]]", "", regex=True),
    errors="coerce"
)
discharge = (
    df[
        (df["type"] == "discharge") &
        (df["Capacity"] > 0.05)
    ]
    .sort_values(["battery_id", "test_id"])
    .copy()
)
print("Total discharge records:", len(discharge))
print("Total batteries:", discharge["battery_id"].nunique())

### **Cell 4 Calculate Initial Capacity & SOH**

In [ ]:
initial_capacity = (
    discharge
    .groupby("battery_id")["Capacity"]
    .first()
)

discharge["Initial_Capacity"] = (
    discharge["battery_id"]
    .map(initial_capacity)
)

discharge["SOH"] = (
    discharge["Capacity"] /
    discharge["Initial_Capacity"] * 100
)

discharge[["battery_id", "test_id", "Capacity", "SOH"]].head()

### **Cell 5 Calculate RUL**

In [ ]:
def calculate_rul(group):

    eol_threshold = 0.70 * group["Initial_Capacity"].iloc[0]

    eol_cycles = group.loc[
        group["Capacity"] <= eol_threshold,
        "test_id"
    ]

    eol_cycle = (
        eol_cycles.iloc[0]
        if not eol_cycles.empty
        else group["test_id"].max()
    )

    group["EOL_Cycle"] = eol_cycle

    group["RUL"] = np.maximum(
        0,
        eol_cycle - group["test_id"]
    )

    return group


discharge = (
    discharge
    .groupby("battery_id", group_keys=False)
    .apply(calculate_rul)
)

discharge[["battery_id", "test_id", "SOH", "RUL"]].head()

### **Cell 6 Capacity Degradation Plot**

In [ ]:
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=discharge,
    x="test_id",
    y="Capacity",
    hue="ambient_temperature",
    units="battery_id",
    estimator=None,
    alpha=0.7
)

plt.title("Battery Capacity Degradation")
plt.xlabel("Cycle Number")
plt.ylabel("Discharge Capacity (Ah)")
plt.legend(title="Temperature (°C)")

plt.tight_layout()
plt.show()

### **Cell 7 SOH Degradation Plot**

In [ ]:
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=discharge,
    x="test_id",
    y="SOH",
    hue="battery_id",
    legend=False,
    alpha=0.5
)

plt.axhline(
    80,
    linestyle="--",
    label="80% SOH"
)

plt.axhline(
    70,
    linestyle="--",
    label="70% SOH"
)

plt.title("State of Health (SOH) Degradation")
plt.xlabel("Cycle Number")
plt.ylabel("SOH (%)")
plt.legend()

plt.tight_layout()
plt.show()

### **Cell 8 RUL Distribution**

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    discharge["RUL"],
    bins=30,
    kde=True
)

plt.title("Remaining Useful Life (RUL) Distribution")
plt.xlabel("RUL (Cycles)")
plt.ylabel("Number of Samples")

plt.tight_layout()
plt.show()

### **Cell 9 Battery Quality Audit**

In [ ]:
discharge["Capacity_Change"] = (
    discharge
    .groupby("battery_id")["Capacity"]
    .diff()
    .abs()
)

audit = (
    discharge
    .groupby("battery_id")
    .agg(
        Total_Cycles=("test_id", "count"),
        Min_Capacity=("Capacity", "min"),
        Max_Capacity=("Capacity", "max"),
        Mean_Capacity_Change=("Capacity_Change", "mean")
    )
    .reset_index()
)

display(audit)

### **Cell 10 Identify Difficult and Anomalous Batteries**

In [ ]:
difficult_cells = audit[
    (audit["Mean_Capacity_Change"] > 0.04) |
    (audit["Total_Cycles"] < 30)
]

print("Potentially Difficult / Anomalous Batteries:")

display(difficult_cells)

### **Cell 11 Final Summary**

In [ ]:
print("Dataset Summary")

print(
    f"Total batteries       : "
    f"{discharge['battery_id'].nunique()}"
)

print(
    f"Total discharge cycles: "
    f"{len(discharge)}"
)

print(
    f"Average initial capacity: "
    f"{discharge['Initial_Capacity'].mean():.3f} Ah"
)

print(
    f"Average SOH           : "
    f"{discharge['SOH'].mean():.2f}%"
)

print(
    f"Average RUL           : "
    f"{discharge['RUL'].mean():.1f} cycles"
)

# Section 2: Sequence Processing & Pipeline Construction

## Day 11: Sequence Processing

*(Source notebook: `day11_Rajnish_Sequence_Processing.ipynb`)*

### ***Cell 1 Imports & Configuration***

In [ ]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", font_scale=1.1)

### ***Cell 2 Mount Google Drive***

In [ ]:
drive.mount('/content/drive')

### ***Cell 3 Dataset Path & Output Directory***

In [ ]:
RANDOM_SEED = 42
PIPELINE_VERSION = "v1.0.0"

np.random.seed(RANDOM_SEED)

DRIVE_ZIP = "/content/drive/MyDrive/cleaned_dataset.zip"
OUTPUT_DIR = "/content/processed_battery_dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError("cleaned_dataset.zip not found in Drive.")

print("Dataset:", DRIVE_ZIP)
print("Output:", OUTPUT_DIR)

### ***Cell 4 Extract Dataset***

In [ ]:
DATASET_DIR = "/content/cleaned_dataset"

if not os.path.exists(DATASET_DIR):
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zip_ref:
        zip_ref.extractall("/content")

print("Dataset extracted successfully.")

### ***Cell 5 Locate Metadata***

In [ ]:
import glob

metadata_files = glob.glob(
    "/content/cleaned_dataset/**/metadata.csv",
    recursive=True
)

if not metadata_files:
    raise FileNotFoundError("metadata.csv not found.")

METADATA_PATH = metadata_files[0]
DATA_CSV_DIR = os.path.join(
    os.path.dirname(METADATA_PATH),
    "data"
)

print("Metadata:", METADATA_PATH)
print("Data directory:", DATA_CSV_DIR)

### ***Cell 6 Load & Clean Metadata***

In [ ]:
df = pd.read_csv(METADATA_PATH)

df["Capacity"] = pd.to_numeric(
    df["Capacity"]
    .astype(str)
    .str.replace(r"[\[\]]", "", regex=True),
    errors="coerce"
)

dis_df = (
    df[
        (df["type"] == "discharge") &
        (df["Capacity"] > 0.05)
    ]
    .sort_values(["battery_id", "test_id"])
    .copy()
)

print("Total records:", len(dis_df))
print("Total batteries:", dis_df["battery_id"].nunique())

display(dis_df.head())

### ***Cell 7 Calculate Initial Capacity & SOH***

In [ ]:
initial_capacity = (
    dis_df
    .groupby("battery_id")["Capacity"]
    .first()
)

dis_df["initial_capacity"] = (
    dis_df["battery_id"]
    .map(initial_capacity)
)

dis_df["SOH"] = (
    dis_df["Capacity"] /
    dis_df["initial_capacity"]
)

print("SOH calculated successfully.")

display(
    dis_df[
        ["battery_id", "test_id", "Capacity", "initial_capacity", "SOH"]
    ].head()
)

### ***Cell 8 Calculate RUL***

In [ ]:
def assign_rul(group):

    eol_threshold = (
        0.70 * group["initial_capacity"].iloc[0]
    )

    eol_cycles = group.loc[
        group["Capacity"] <= eol_threshold,
        "test_id"
    ]

    if len(eol_cycles) > 0:
        eol_cycle = eol_cycles.iloc[0]
    else:
        eol_cycle = group["test_id"].max()

    group["EOL_cycle"] = eol_cycle

    group["RUL"] = np.maximum(
        0,
        eol_cycle - group["test_id"]
    )

    return group


dis_df = (
    dis_df
    .groupby("battery_id", group_keys=False)
    .apply(assign_rul)
)

print("RUL calculated successfully.")

display(
    dis_df[
        ["battery_id", "test_id", "Capacity", "SOH", "EOL_cycle", "RUL"]
    ].head()
)

### ***Cell 9 Define Time Grid & Channels***

In [ ]:
GRID_DT = 10
GRID_MAX_T = 500

TIME_GRID = np.arange(
    0,
    GRID_MAX_T + GRID_DT,
    GRID_DT
)

GRID_LEN = len(TIME_GRID)

CHANNELS = [
    "Voltage_measured",
    "Current_measured",
    "Temperature_measured"
]

print("Time steps per cycle:", GRID_LEN)
print("Channels:", CHANNELS)

### ***Cell 10 Align All Discharge Cycles***

In [ ]:
aligned_cycles = []
meta_records = []

for _, row in dis_df.iterrows():

    file_path = os.path.join(
        DATA_CSV_DIR,
        row["filename"]
    )

    if not os.path.exists(file_path):
        continue

    try:
        raw = pd.read_csv(file_path)

        if (
            len(raw) < 10 or
            "Time" not in raw.columns or
            raw["Time"].iloc[-1] < 60
        ):
            continue

        raw = (
            raw
            .drop_duplicates(subset=["Time"])
            .sort_values("Time")
        )

        t_raw = raw["Time"].values

        cycle_tensor = np.zeros(
            (GRID_LEN, len(CHANNELS)),
            dtype=np.float32
        )

        valid = True

        for channel_idx, channel in enumerate(CHANNELS):

            if channel not in raw.columns:
                valid = False
                break

            interpolation = interp1d(
                t_raw,
                raw[channel].values,
                kind="linear",
                bounds_error=False,
                fill_value="extrapolate"
            )

            cycle_tensor[:, channel_idx] = (
                interpolation(TIME_GRID)
            )

        if not valid:
            continue

        aligned_cycles.append(cycle_tensor)

        meta_records.append({
            "battery_id": row["battery_id"],
            "test_id": int(row["test_id"]),
            "ambient_temperature": row["ambient_temperature"],
            "Capacity": float(row["Capacity"]),
            "SOH": float(row["SOH"]),
            "RUL": int(row["RUL"]),
            "filename": row["filename"]
        })

    except Exception:
        continue

aligned_X = np.stack(aligned_cycles, axis=0)
meta_df = pd.DataFrame(meta_records)

print("Aligned tensor shape:", aligned_X.shape)
print("Metadata shape:", meta_df.shape)

### ***Cell 11 Define Test Batteries***

In [ ]:
test_cells = [
    "B0018",
    "B0028",
    "B0032",
    "B0036",
    "B0040",
    "B0043",
    "B0044",
    "B0051",
    "B0056"
]

train_indices = meta_df[
    ~meta_df["battery_id"].isin(test_cells)
].index.values

print("Training samples:", len(train_indices))
print("Test batteries:", len(test_cells))
print("Test cells:", test_cells)

### ***Cell 12 Fit StandardScaler on Training Batteries Only***

In [ ]:
scaler = StandardScaler()

train_data = aligned_X[
    train_indices
].reshape(-1, len(CHANNELS))

scaler.fit(train_data)

print("Scaler fitted on training batteries only.")
print("Scaler mean:", scaler.mean_)
print("Scaler scale:", scaler.scale_)

### ***Cell 13 Scale Complete Dataset***

In [ ]:
aligned_X_scaled = np.zeros_like(aligned_X)

for i in range(len(aligned_X)):

    aligned_X_scaled[i] = scaler.transform(
        aligned_X[i]
    )

print("Scaling completed.")
print("Scaled tensor shape:", aligned_X_scaled.shape)

### ***Cell 14 Create Sliding Windows***

In [ ]:
WINDOW_SIZE = 10

windowed_X = []
windowed_y_soh = []
windowed_y_rul = []
windowed_y_cap = []
windowed_meta = []

for battery_id, group in meta_df.groupby("battery_id"):

    group_indices = group.index.values

    if len(group_indices) < WINDOW_SIZE:
        continue

    for start in range(
        len(group_indices) - WINDOW_SIZE + 1
    ):

        end = start + WINDOW_SIZE

        target_idx = group_indices[end - 1]

        window_tensor = aligned_X_scaled[
            group_indices[start:end]
        ]

        windowed_X.append(window_tensor)

        windowed_y_soh.append(
            meta_df.loc[target_idx, "SOH"]
        )

        windowed_y_rul.append(
            meta_df.loc[target_idx, "RUL"]
        )

        windowed_y_cap.append(
            meta_df.loc[target_idx, "Capacity"]
        )

        windowed_meta.append({
            "battery_id": battery_id,
            "target_test_id": int(
                meta_df.loc[target_idx, "test_id"]
            ),
            "ambient_temperature": meta_df.loc[
                target_idx,
                "ambient_temperature"
            ],
            "window_start_idx": start,
            "window_end_idx": end - 1
        })

windowed_X = np.stack(windowed_X)

windowed_y_soh = np.array(
    windowed_y_soh,
    dtype=np.float32
)

windowed_y_rul = np.array(
    windowed_y_rul,
    dtype=np.float32
)

windowed_y_cap = np.array(
    windowed_y_cap,
    dtype=np.float32
)

windowed_meta_df = pd.DataFrame(
    windowed_meta
)

print("Sliding-window shape:", windowed_X.shape)

### ***Cell 15 Save Aligned Cycle Dataset***

In [ ]:
aligned_path = os.path.join(
    OUTPUT_DIR,
    f"battery_aligned_cycles_{PIPELINE_VERSION}.npz"
)

np.savez_compressed(
    aligned_path,
    X=aligned_X_scaled,
    y_soh=meta_df["SOH"].values.astype(np.float32),
    y_rul=meta_df["RUL"].values.astype(np.float32),
    y_cap=meta_df["Capacity"].values.astype(np.float32)
)

print("Saved:", aligned_path)

### ***Cell 16 Save Sliding Window Dataset***

In [ ]:
window_path = os.path.join(
    OUTPUT_DIR,
    f"battery_sliding_windows_{PIPELINE_VERSION}.npz"
)

np.savez_compressed(
    window_path,
    X_windows=windowed_X,
    y_soh=windowed_y_soh,
    y_rul=windowed_y_rul,
    y_cap=windowed_y_cap
)

print("Saved:", window_path)

### ***Cell 17 Save Metadata***

In [ ]:
aligned_meta_path = os.path.join(
    OUTPUT_DIR,
    f"aligned_cycles_metadata_{PIPELINE_VERSION}.csv"
)

window_meta_path = os.path.join(
    OUTPUT_DIR,
    f"sliding_windows_metadata_{PIPELINE_VERSION}.csv"
)

meta_df.to_csv(
    aligned_meta_path,
    index=False
)

windowed_meta_df.to_csv(
    window_meta_path,
    index=False
)

print("Metadata files saved.")

### ***Cell 18 Save Pipeline Configuration***

In [ ]:
config = {
    "version": PIPELINE_VERSION,
    "seed": RANDOM_SEED,

    "grid_dt_seconds": GRID_DT,
    "grid_max_time_seconds": GRID_MAX_T,
    "time_steps_per_cycle": GRID_LEN,

    "channels": CHANNELS,

    "scaling": "StandardScaler_fit_on_train_cells_only",

    "window_size_cycles": WINDOW_SIZE,

    "total_aligned_cycles": int(
        len(aligned_X)
    ),

    "total_sliding_windows": int(
        len(windowed_X)
    ),

    "aligned_tensor_shape": list(
        aligned_X.shape
    ),

    "windowed_tensor_shape": list(
        windowed_X.shape
    ),

    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist()
}

config_path = os.path.join(
    OUTPUT_DIR,
    f"pipeline_config_{PIPELINE_VERSION}.json"
)

with open(config_path, "w") as file:
    json.dump(
        config,
        file,
        indent=4
    )

print("Configuration saved:")
print(config_path)

### ***Cell 19 Final Verification***

In [ ]:
print("=" * 60)
print("PREPROCESSING PIPELINE COMPLETE")
print("=" * 60)

print(f"Aligned cycles      : {len(aligned_X)}")
print(f"Aligned shape       : {aligned_X.shape}")

print(f"Sliding windows     : {len(windowed_X)}")
print(f"Window shape        : {windowed_X.shape}")

print(f"Training samples    : {len(train_indices)}")
print(f"Test batteries      : {len(test_cells)}")

print("\nSaved files:")

for file in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", file)

### ***Cell 20 Battery Cycle Visualization***

In [ ]:
battery_id = "B0005"

battery_meta = meta_df[
    meta_df["battery_id"] == battery_id
].sort_values("test_id")

selected_indices = [
    battery_meta.index[0],
    battery_meta.index[len(battery_meta) // 2],
    battery_meta.index[-1]
]

labels = ["Early Cycle", "Middle Cycle", "Late Cycle"]

fig, axes = plt.subplots(3, 1, figsize=(12, 12))

for idx, label in zip(selected_indices, labels):

    cycle = aligned_X[idx]

    axes[0].plot(
        TIME_GRID,
        cycle[:, 0],
        label=label
    )

    axes[1].plot(
        TIME_GRID,
        cycle[:, 1],
        label=label
    )

    axes[2].plot(
        TIME_GRID,
        cycle[:, 2],
        label=label
    )

axes[0].set_title(f"Voltage Profile — {battery_id}")
axes[0].set_ylabel("Voltage (V)")
axes[0].legend()

axes[1].set_title(f"Current Profile — {battery_id}")
axes[1].set_ylabel("Current (A)")
axes[1].legend()

axes[2].set_title(f"Temperature Profile — {battery_id}")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Temperature (°C)")
axes[2].legend()

plt.tight_layout()
plt.show()

### ***Cell 21 SOH & RUL Visualization***

In [ ]:
battery_id = "B0005"

battery_data = meta_df[
    meta_df["battery_id"] == battery_id
].sort_values("test_id")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    battery_data["test_id"],
    battery_data["SOH"],
    marker="o"
)

axes[0].axhline(
    0.70,
    linestyle="--",
    label="70% EOL"
)

axes[0].set_title(f"SOH Degradation — {battery_id}")
axes[0].set_xlabel("Cycle Number")
axes[0].set_ylabel("SOH")
axes[0].legend()

axes[1].plot(
    battery_data["test_id"],
    battery_data["RUL"],
    marker="o"
)

axes[1].set_title(f"RUL Degradation — {battery_id}")
axes[1].set_xlabel("Cycle Number")
axes[1].set_ylabel("RUL (Cycles)")

plt.tight_layout()
plt.show()

### ***Cell 22 Dataset Tensor Visualization***

In [ ]:
print("PREPROCESSED DATA")

print("Aligned X shape :", aligned_X_scaled.shape)
print("Window X shape  :", windowed_X.shape)

print("\nTargets:")
print("SOH shape       :", windowed_y_soh.shape)
print("RUL shape       :", windowed_y_rul.shape)
print("Capacity shape  :", windowed_y_cap.shape)

## Day 12: Sequence Pipeline (Train/Val/Test, Leakage-Safe)

*(Source notebook: `Day12_Rajnish_SequencePipeline.ipynb`)*

### ***Cell 1 Imports & Configuration***

In [ ]:
from google.colab import drive
import os
import json
import numpy as np
import pandas as pd

from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

drive.mount('/content/drive')

RANDOM_SEED = 42
DATASET_VERSION = "V1_FROZEN"

np.random.seed(RANDOM_SEED)

OUTPUT_DIR = f"/content/dataset_{DATASET_VERSION.lower()}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DRIVE_ZIP = "/content/drive/MyDrive/cleaned_dataset.zip"

### ***Cell 2 Unzip Dataset***

In [ ]:
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(f"Dataset not found: {DRIVE_ZIP}")

!unzip -o -q "/content/drive/MyDrive/cleaned_dataset.zip" -d /content/

if os.path.exists("/content/cleaned_dataset"):
    DATASET_DIR = "/content/cleaned_dataset"
else:
    DATASET_DIR = "/content"

print("Dataset Directory:", DATASET_DIR)

### ***Cell 3 Locate Metadata and Data Folder***

In [ ]:
METADATA_PATH = os.path.join(DATASET_DIR, "metadata.csv")
DATA_CSV_DIR = os.path.join(DATASET_DIR, "data")

if not os.path.exists(METADATA_PATH):
    raise FileNotFoundError("metadata.csv not found")

if not os.path.exists(DATA_CSV_DIR):
    raise FileNotFoundError("data folder not found")

print("Metadata:", METADATA_PATH)
print("Data Directory:", DATA_CSV_DIR)

### ***Cell 4 Load and Prepare Metadata***

In [ ]:
df = pd.read_csv(METADATA_PATH)

df["Capacity"] = pd.to_numeric(
    df["Capacity"].astype(str).str.replace(r"[\[\]]", "", regex=True),
    errors="coerce"
)

dis_df = df[
    (df["type"] == "discharge") &
    (df["Capacity"] > 0.05)
].sort_values(
    ["battery_id", "test_id"]
).copy()

print("Total metadata rows:", len(df))
print("Discharge cycles:", len(dis_df))
print("Unique batteries:", dis_df["battery_id"].nunique())

### ***Cell 5 Calculate SOH and RUL***

In [ ]:
c0_map = (
    dis_df
    .groupby("battery_id")["Capacity"]
    .first()
    .to_dict()
)

dis_df["initial_capacity"] = dis_df["battery_id"].map(c0_map)
dis_df["SOH"] = dis_df["Capacity"] / dis_df["initial_capacity"]

In [ ]:
def assign_rul(sub):
    eol_thresh = 0.70 * sub["initial_capacity"].iloc[0]

    eol_sub = sub[sub["Capacity"] <= eol_thresh]

    if len(eol_sub) > 0:
        eol_cycle = eol_sub["test_id"].iloc[0]
    else:
        eol_cycle = sub["test_id"].max()

    sub = sub.copy()
    sub["EOL_cycle"] = eol_cycle
    sub["RUL"] = np.maximum(
        0,
        eol_cycle - sub["test_id"]
    )

    return sub


dis_df = (
    dis_df
    .groupby("battery_id", group_keys=False)
    .apply(assign_rul)
)

### ***Cell 6 Define Time Grid***

In [ ]:
GRID_DT = 10
GRID_MAX_T = 500

TIME_GRID = np.arange(
    0,
    GRID_MAX_T + GRID_DT,
    GRID_DT
)

GRID_LEN = len(TIME_GRID)

CHANNELS = [
    "Voltage_measured",
    "Current_measured",
    "Temperature_measured"
]

print("Time steps:", GRID_LEN)
print("Channels:", CHANNELS)

### ***Cell 7 Align All Battery Cycles***

In [ ]:
aligned_cycles = []
meta_records = []

for _, r in dis_df.iterrows():

    fpath = os.path.join(
        DATA_CSV_DIR,
        r["filename"]
    )

    if not os.path.exists(fpath):
        continue

    try:
        raw = pd.read_csv(fpath)

        if (
            len(raw) < 10 or
            "Time" not in raw.columns or
            raw["Time"].iloc[-1] < 60
        ):
            continue

        raw = (
            raw
            .drop_duplicates(subset=["Time"])
            .sort_values("Time")
        )

        t_raw = raw["Time"].values

        cycle_tensor = np.zeros(
            (GRID_LEN, len(CHANNELS)),
            dtype=np.float32
        )

        valid = True

        for c_idx, col in enumerate(CHANNELS):

            if col not in raw.columns:
                valid = False
                break

            interp_fn = interp1d(
                t_raw,
                raw[col].values,
                kind="linear",
                bounds_error=False,
                fill_value="extrapolate"
            )

            cycle_tensor[:, c_idx] = interp_fn(
                TIME_GRID
            )

        if not valid:
            continue

        aligned_cycles.append(cycle_tensor)

        meta_records.append({
            "battery_id": r["battery_id"],
            "test_id": int(r["test_id"]),
            "ambient_temperature": r["ambient_temperature"],
            "Capacity": float(r["Capacity"]),
            "SOH": float(r["SOH"]),
            "RUL": int(r["RUL"]),
            "filename": r["filename"]
        })

    except Exception:
        continue

### ***Cell 8 Create Aligned Dataset***

In [ ]:
if len(aligned_cycles) == 0:
    raise RuntimeError("No valid cycles were processed.")

aligned_X = np.stack(
    aligned_cycles,
    axis=0
)

meta_df = pd.DataFrame(
    meta_records
).reset_index(drop=True)

print("Aligned X shape:", aligned_X.shape)
print("Metadata shape:", meta_df.shape)

### ***Cell 9 Define Cell-Level Train/Validation/Test Split***

In [ ]:
test_cells = [
    "B0018", "B0028", "B0032",
    "B0036", "B0040", "B0043",
    "B0044", "B0051", "B0056"
]

val_cells = [
    "B0007", "B0027", "B0031",
    "B0034", "B0039", "B0042",
    "B0048", "B0050", "B0054"
]

train_cells = [
    b for b in meta_df["battery_id"].unique()
    if b not in test_cells
    and b not in val_cells
]

print("Train cells:", len(train_cells))
print("Validation cells:", len(val_cells))
print("Test cells:", len(test_cells))

### ***Cell 10 Leakage-Safe Normalization***

In [ ]:
train_mask = meta_df["battery_id"].isin(
    train_cells
).values

scaler = StandardScaler()

scaler.fit(
    aligned_X[train_mask]
    .reshape(-1, len(CHANNELS))
)

aligned_X_scaled = np.zeros_like(
    aligned_X
)

for i in range(len(aligned_X)):
    aligned_X_scaled[i] = scaler.transform(
        aligned_X[i]
    )

print("Normalization completed.")
print("Scaler means:", scaler.mean_)
print("Scaler scales:", scaler.scale_)

### ***Cell 11 Create Sliding Windows***

In [ ]:
WINDOW_SIZE = 10

windowed_X = []
windowed_y_soh = []
windowed_y_rul = []
windowed_y_cap = []
windowed_meta = []

for b_id, grp in meta_df.groupby("battery_id"):

    grp_indices = grp.index.values

    if len(grp_indices) < WINDOW_SIZE:
        continue

    for start in range(
        len(grp_indices) - WINDOW_SIZE + 1
    ):

        end = start + WINDOW_SIZE
        target_idx = grp_indices[end - 1]

        windowed_X.append(
            aligned_X_scaled[
                grp_indices[start:end]
            ]
        )

        windowed_y_soh.append(
            meta_df.loc[target_idx, "SOH"]
        )

        windowed_y_rul.append(
            meta_df.loc[target_idx, "RUL"]
        )

        windowed_y_cap.append(
            meta_df.loc[target_idx, "Capacity"]
        )

        windowed_meta.append({
            "battery_id": b_id,
            "target_test_id": int(
                meta_df.loc[target_idx, "test_id"]
            ),
            "ambient_temperature":
                meta_df.loc[
                    target_idx,
                    "ambient_temperature"
                ],
            "window_start_idx": start,
            "window_end_idx": end - 1
        })

### ***Cell 12 Convert Windows to NumPy Arrays***

In [ ]:
windowed_X = np.stack(
    windowed_X,
    axis=0
)

windowed_y_soh = np.array(
    windowed_y_soh,
    dtype=np.float32
)

windowed_y_rul = np.array(
    windowed_y_rul,
    dtype=np.float32
)

windowed_y_cap = np.array(
    windowed_y_cap,
    dtype=np.float32
)

windowed_meta_df = pd.DataFrame(
    windowed_meta
).reset_index(drop=True)

print("Windowed X:", windowed_X.shape)
print("SOH:", windowed_y_soh.shape)
print("RUL:", windowed_y_rul.shape)
print("Capacity:", windowed_y_cap.shape)

### ***Cell 13 Create Train/Validation/Test Windows***

In [ ]:
win_train_mask = (
    windowed_meta_df["battery_id"]
    .isin(train_cells)
    .values
)

win_val_mask = (
    windowed_meta_df["battery_id"]
    .isin(val_cells)
    .values
)

win_test_mask = (
    windowed_meta_df["battery_id"]
    .isin(test_cells)
    .values
)

### ***Cell 14 Leakage and Data Integrity Checks***

In [ ]:
assert len(set(train_cells) & set(val_cells)) == 0
assert len(set(train_cells) & set(test_cells)) == 0
assert len(set(val_cells) & set(test_cells)) == 0

assert not np.isnan(windowed_X).any()
assert not np.isnan(windowed_y_soh).any()
assert not np.isnan(windowed_y_rul).any()

assert (
    set(windowed_meta_df.loc[
        win_train_mask, "battery_id"
    ]).issubset(set(train_cells))
)

assert (
    set(windowed_meta_df.loc[
        win_val_mask, "battery_id"
    ]).issubset(set(val_cells))
)

assert (
    set(windowed_meta_df.loc[
        win_test_mask, "battery_id"
    ]).issubset(set(test_cells))
)

print("All leakage and integrity checks PASSED.")

### ***Cell 15 Save Train/Validation/Test Dataset***

In [ ]:
np.savez_compressed(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_cell_splits.npz"
    ),
    X_train=windowed_X[win_train_mask],
    y_train_soh=windowed_y_soh[win_train_mask],
    y_train_rul=windowed_y_rul[win_train_mask],
    X_val=windowed_X[win_val_mask],
    y_val_soh=windowed_y_soh[win_val_mask],
    y_val_rul=windowed_y_rul[win_val_mask],
    X_test=windowed_X[win_test_mask],
    y_test_soh=windowed_y_soh[win_test_mask],
    y_test_rul=windowed_y_rul[win_test_mask]
)

print("Cell split dataset saved.")

### ***Cell 16 Save Full Window Dataset***

In [ ]:
np.savez_compressed(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_full_windows.npz"
    ),
    X=windowed_X,
    y_soh=windowed_y_soh,
    y_rul=windowed_y_rul,
    y_cap=windowed_y_cap
)

print("Full window dataset saved.")

### ***Cell 17 Save Metadata***

In [ ]:
meta_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_cycles_metadata.csv"
    ),
    index=False
)

windowed_meta_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_windows_metadata.csv"
    ),
    index=False
)

print("Metadata files saved.")

### ***Cell 18 Create Dataset Manifest***

In [ ]:
manifest = {
    "status": "APPROVED_AND_FROZEN",
    "dataset_version": DATASET_VERSION,
    "timestamp_frozen": pd.Timestamp.now().isoformat(),
    "random_seed": RANDOM_SEED,

    "defense_parameters": {
        "observation_window_seconds": GRID_MAX_T,
        "sampling_dt_seconds": GRID_DT,
        "time_steps_per_cycle": GRID_LEN,
        "channels": CHANNELS,
        "sliding_window_cycles": WINDOW_SIZE,
        "normalization_defense":
            "StandardScaler fit exclusively on training cells",
        "rul_definition":
            "Remaining cycles until capacity drops below 70% of initial C0",
        "soh_definition":
            "Capacity_k / Initial_Capacity_C0"
    },

    "shapes": {
        "full_windowed_tensor":
            list(windowed_X.shape),
        "train_windows":
            int(win_train_mask.sum()),
        "val_windows":
            int(win_val_mask.sum()),
        "test_windows":
            int(win_test_mask.sum())
    },

    "cell_split_distribution": {
        "train_cells": sorted(train_cells),
        "val_cells": sorted(val_cells),
        "test_cells": sorted(test_cells)
    },

    "scaler_statistics": {
        "channel_means":
            scaler.mean_.tolist(),
        "channel_scales":
            scaler.scale_.tolist()
    }
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_manifest.json"
    ),
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=4
    )

print("Manifest created.")

### ***Cell 19 Final Dataset Freeze Summary***

In [ ]:
print("=" * 60)
print("DATASET VERSION 1 FREEZE SUMMARY")
print("=" * 60)

print(f"Status: {manifest['status']}")
print(f"Total Cycles Processed: {len(aligned_X)}")
print(
    f"Total Sliding Windows (W={WINDOW_SIZE}): "
    f"{len(windowed_X)}"
)

print(
    "Tensor Shape: "
    f"(Windows={windowed_X.shape[0]}, "
    f"Cycles={windowed_X.shape[1]}, "
    f"Steps={windowed_X.shape[2]}, "
    f"Channels={windowed_X.shape[3]})"
)

print(
    f"Train Windows: {win_train_mask.sum()} "
    f"({len(train_cells)} cells)"
)

print(
    f"Val Windows:   {win_val_mask.sum()} "
    f"({len(val_cells)} cells)"
)

print(
    f"Test Windows:  {win_test_mask.sum()} "
    f"({len(test_cells)} cells)"
)

print("Leakage Checks: PASSED")
print("Normalization: Training cells only")
print(f"Output Directory: {OUTPUT_DIR}")
print("=" * 60)

### ***Cell 20 Verify Saved Files***

In [ ]:
print("Generated files:")

for file in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, file)
    print(
        f"{file:<45} "
        f"{os.path.getsize(path) / (1024**2):.2f} MB"
    )

# Section 3: Baseline Prognostics & Key Feature Analysis

## Day 13: Baseline Prognostic Models (SoH & RUL)

*(Source notebook: `Day13_Rajnish_BaselineProganstic.ipynb`)*

### Battery Health Prognostics Baseline Evaluation (SoH & RUL)
This notebook evaluates two **baseline machine-learning models**

- **SoH (State of Health)** remaining capacity as a fraction of initial capacity
- **RUL (Remaining Useful Life)** number of cycles left before end-of-life (EOL)

The workflow is:

1. Load the frozen, leakage safe dataset
2. Train baseline Ridge and Random Forest models on the training cells
3. Evaluate on validation cells and **held-out, unseen test cells**
4. Visualize predicted-vs-actual performance
5. Rank test cells by prediction difficulty

#### 1. Setup & Imports
Standard data-science stack: `pandas`/`numpy` for data handling, `scikit-learn` for modelling and metrics, `matplotlib`/`seaborn` for plots.

In [ ]:
import os, glob, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

#### 2. Load the Dataset

In [ ]:
v1_dir = "/content/dataset_v1_frozen"
data_file = os.path.join(v1_dir, "dataset_v1_cell_splits.npz")
meta_file = os.path.join(v1_dir, "dataset_v1_windows_metadata.csv")

if not os.path.exists(data_file):
    from google.colab import drive
    drive.mount('/content/drive')
    meta_path = [os.path.join(r, f) for r, d, fs in os.walk('/content') for f in fs if f == 'metadata.csv' and 'drive' not in r]
    if not meta_path:
        drive_zips = glob.glob('/content/drive/MyDrive/**/cleaned_dataset*.zip', recursive=True)
        if drive_zips:
            !unzip -o -q "{drive_zips[0]}" -d /content/
        meta_path = [os.path.join(r, f) for r, d, fs in os.walk('/content') for f in fs if f == 'metadata.csv' and 'drive' not in r]

    METADATA_PATH = meta_path[0]
    DATA_CSV_DIR = os.path.join(os.path.dirname(METADATA_PATH), 'data')
    df = pd.read_csv(METADATA_PATH)
    df['Capacity'] = pd.to_numeric(df['Capacity'].astype(str).str.replace(r'[\[\]]', '', regex=True), errors='coerce')
    dis_df = df[(df['type'] == 'discharge') & (df['Capacity'] > 0.05)].sort_values(['battery_id', 'test_id']).copy()

    c0 = dis_df.groupby('battery_id')['Capacity'].first()
    dis_df['initial_capacity'] = dis_df['battery_id'].map(c0)
    dis_df['SOH'] = dis_df['Capacity'] / dis_df['initial_capacity']

    def assign_rul(sub):
        eol = sub[sub['Capacity'] <= 0.70 * sub['initial_capacity'].iloc[0]]
        eol_cycle = eol['test_id'].iloc[0] if len(eol) > 0 else sub['test_id'].max()
        sub['RUL'] = np.maximum(0, eol_cycle - sub['test_id'])
        return sub
    dis_df = dis_df.groupby('battery_id', group_keys=False).apply(assign_rul)

    records = []
    for idx, r in dis_df.iterrows():
        f = os.path.join(DATA_CSV_DIR, r['filename'])
        if not os.path.exists(f): continue
        try:
            raw = pd.read_csv(f)
            if len(raw) < 10 or 'Time' not in raw or raw['Time'].iloc[-1] < 60: continue
            w = raw[raw['Time'] <= 500]
            if len(w) < 5: continue
            dt = w['Time'].iloc[-1] - w['Time'].iloc[0] + 1e-6
            records.append({
                'battery_id': r['battery_id'], 'test_id': int(r['test_id']),
                'ambient_temperature': r['ambient_temperature'],
                'feat_R_est': (w['Voltage_measured'].iloc[0] - w['Voltage_measured'].iloc[min(3, len(w)-1)]) / (abs(w['Current_measured'].iloc[min(3, len(w)-1)]) + 1e-6),
                'feat_dV_dt': (w['Voltage_measured'].iloc[-1] - w['Voltage_measured'].iloc[0]) / dt,
                'feat_dT_dt': (w['Temperature_measured'].iloc[-1] - w['Temperature_measured'].iloc[0]) / dt,
                'feat_V_mean': float(w['Voltage_measured'].mean()),
                'feat_T_mean': float(w['Temperature_measured'].mean()),
                'target_SOH': float(r['SOH']), 'target_RUL': float(r['RUL'])
            })
        except: pass
    feat_df = pd.DataFrame(records)
    test_cells = ['B0018', 'B0028', 'B0032', 'B0036', 'B0040', 'B0043', 'B0044', 'B0051', 'B0056']
    val_cells = ['B0007', 'B0027', 'B0031', 'B0034', 'B0039', 'B0042', 'B0048', 'B0050', 'B0054']

    train_df = feat_df[~feat_df['battery_id'].isin(test_cells + val_cells)]
    val_df = feat_df[feat_df['battery_id'].isin(val_cells)]
    test_df = feat_df[feat_df['battery_id'].isin(test_cells)]

    feat_cols = [c for c in feat_df.columns if c.startswith('feat_')]
    X_train, y_train_soh, y_train_rul = train_df[feat_cols].values, train_df['target_SOH'].values, train_df['target_RUL'].values
    X_val, y_val_soh, y_val_rul = val_df[feat_cols].values, val_df['target_SOH'].values, val_df['target_RUL'].values
    X_test, y_test_soh, y_test_rul = test_df[feat_cols].values, test_df['target_SOH'].values, test_df['target_RUL'].values
    test_meta = test_df[['battery_id', 'test_id', 'ambient_temperature']].reset_index(drop=True)
else:
    data = np.load(data_file)
    X_train = data['X_train'].reshape(len(data['X_train']), -1)
    y_train_soh = data['y_train_soh']
    y_train_rul = data['y_train_rul']
    X_val = data['X_val'].reshape(len(data['X_val']), -1)
    y_val_soh = data['y_val_soh']
    y_val_rul = data['y_val_rul']
    X_test = data['X_test'].reshape(len(data['X_test']), -1)
    y_test_soh = data['y_test_soh']
    y_test_rul = data['y_test_rul']
    meta_df = pd.read_csv(meta_file)
    test_cells = ['B0018', 'B0028', 'B0032', 'B0036', 'B0040', 'B0043', 'B0044', 'B0051', 'B0056']
    test_meta = meta_df[meta_df['battery_id'].isin(test_cells)].reset_index(drop=True)

#### 3. Evaluation Metrics
A small helper that returns the four standard regression metrics used throughout the project: **MAE**, **RMSE**, **MAPE (%)**, and **R²**.

In [ ]:
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-3, None))) * 100
    r2 = r2_score(y_true, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'MAPE(%)': mape, 'R2': r2}

#### 4. Train Baseline Models
Two families of baseline model, one for each target:

- **Ridge Regression** —> a simple, well-regularized linear baseline
- **Random Forest Regressor** —> a stronger non-linear baseline (100 trees)

Both are trained only on `X_train` / the training cells' targets. The Random Forest predictions are what we evaluate below.

In [ ]:
models = {
    'SOH_Ridge': Ridge(alpha=1.0).fit(X_train, y_train_soh),
    'SOH_RF': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train_soh),
    'RUL_Ridge': Ridge(alpha=1.0).fit(X_train, y_train_rul),
    'RUL_RF': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train_rul)
}

#### 5. Generate Predictions
Predict SoH and RUL on both the validation split and the fully unseen test cells, using the Random Forest models. RUL predictions are clipped at zero (a cell can't have negative remaining life).

In [ ]:
pred_val_soh = models['SOH_RF'].predict(X_val)
pred_test_soh = models['SOH_RF'].predict(X_test)
pred_val_rul = np.maximum(0, models['RUL_RF'].predict(X_val))
pred_test_rul = np.maximum(0, models['RUL_RF'].predict(X_test))

#### 6. Results Table
Metrics for both targets (**SoH**, **RUL**), on both splits (**Validation**, **Test unseen cells**). The test-cell numbers are the ones that matter most: they show how well the model generalizes to batteries it never saw during training.

In [ ]:
results = [
    {'Target': 'SoH', 'Split': 'Validation', **calc_metrics(y_val_soh, pred_val_soh)},
    {'Target': 'SoH', 'Split': 'Test (Unseen Cells)', **calc_metrics(y_test_soh, pred_test_soh)},
    {'Target': 'RUL', 'Split': 'Validation', **calc_metrics(y_val_rul, pred_val_rul)},
    {'Target': 'RUL', 'Split': 'Test (Unseen Cells)', **calc_metrics(y_test_rul, pred_test_rul)}
]

metrics_df = pd.DataFrame(results)
print("="*65)
print("BASELINE PROGNOSTICS EVALUATION METRICS")
print("="*65)
display(metrics_df)

#### 7. Predicted vs. Actual Test Cells

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test_soh, pred_test_soh, alpha=0.5, color='royalblue', edgecolors='none')
min_s, max_s = min(y_test_soh.min(), pred_test_soh.min()), max(y_test_soh.max(), pred_test_soh.max())
axes[0].plot([min_s, max_s], [min_s, max_s], 'r--', lw=2, label='Ideal y=x')
axes[0].set_title(f"SoH Predicted vs Actual (Test Cells) | R²: {calc_metrics(y_test_soh, pred_test_soh)['R2']:.3f}")
axes[0].set_xlabel("Actual SoH")
axes[0].set_ylabel("Predicted SoH")
axes[0].legend()

axes[1].scatter(y_test_rul, pred_test_rul, alpha=0.5, color='darkorange', edgecolors='none')
min_r, max_r = min(y_test_rul.min(), pred_test_rul.min()), max(y_test_rul.max(), pred_test_rul.max())
axes[1].plot([min_r, max_r], [min_r, max_r], 'r--', lw=2, label='Ideal y=x')
axes[1].set_title(f"RUL Predicted vs Actual (Test Cells) | MAE: {calc_metrics(y_test_rul, pred_test_rul)['MAE']:.1f} Cycles")
axes[1].set_xlabel("Actual RUL (Cycles)")
axes[1].set_ylabel("Predicted RUL (Cycles)")
axes[1].legend()

plt.show()

#### 8. Per Cell Modelling Difficulty

In [ ]:
test_meta['Actual_SOH'] = y_test_soh
test_meta['Pred_SOH'] = pred_test_soh
test_meta['SOH_Error'] = np.abs(y_test_soh - pred_test_soh)
test_meta['Actual_RUL'] = y_test_rul
test_meta['Pred_RUL'] = pred_test_rul
test_meta['RUL_Error'] = np.abs(y_test_rul - pred_test_rul)

cell_difficulty = test_meta.groupby('battery_id').agg({
    'ambient_temperature': 'first',
    'SOH_Error': 'mean',
    'RUL_Error': 'mean',
    'Actual_RUL': 'count'
}).rename(columns={'Actual_RUL': 'Cycle_Count'}).sort_values(by='RUL_Error', ascending=False).reset_index()

print("="*65)
print("MODELLING DIFFICULTY RANKING BY TEST CELL")
print("="*65)
display(cell_difficulty)

#### 9. RUL Error by Test Cell Bar Chart
A quick visual ranking of which unseen test cells the model struggles with most, in terms of mean absolute RUL error (in cycles).

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=cell_difficulty, x='battery_id', y='RUL_Error', palette='Reds_r')
plt.title("RUL Prediction Mean Absolute Error Across Unseen Test Cells")
plt.xlabel("Battery ID")
plt.ylabel("Mean Absolute Error (Cycles)")
plt.show()

#### Summary
- Two baseline models (Ridge, Random Forest) were trained for **SoH** and **RUL** prediction using a small set of hand-crafted per-window features.
- Evaluation followed a strict **group split by `battery_id`** — the test cells were never seen in training, giving a realistic estimate of generalization.
- The metrics table, scatter plots, and per-cell difficulty ranking together give both an aggregate and a cell-by-cell view of baseline performance, forming a reference point against which the Day 5 deep-learning models (LSTM/GRU, 1D-CNN, Transformer) can be compared.

## Day 14: Key Change / Capacity Degradation Analysis

*(Source notebook: `Day14_Rajnish_KeyChange.ipynb`)*

### ***Cell 1 Imports & Output Directory***

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

OUTPUT_DIR = "/content/Week_2_Package"
VIS_DIR = os.path.join(OUTPUT_DIR, "visualizations")

os.makedirs(VIS_DIR, exist_ok=True)

### ***Cell 2 Fetch Dataset from Drive***

In [ ]:
from google.colab import drive
import glob, zipfile

drive.mount("/content/drive")

zip_matches = glob.glob("/content/drive/MyDrive/**/cleaned_dataset.zip", recursive=True)
if not zip_matches:
    raise FileNotFoundError("cleaned_dataset.zip not found in Drive")

with zipfile.ZipFile(zip_matches[0], "r") as zf:
    zf.extractall("/content")

print(f"Extracted: {zip_matches[0]}")

### ***Cell 3 Load metadata.csv***

In [ ]:
import os

meta_files = []

for root, dirs, files_ in os.walk("/content"):
    for file in files_:
        if file.lower() == "metadata.csv":
            meta_files.append(os.path.join(root, file))

print("Found metadata files:")

for path in meta_files:
    print(path)

In [ ]:
import pandas as pd

meta_path = "/content/cleaned_dataset/metadata.csv"

df = pd.read_csv(meta_path)

print("Metadata path:", meta_path)
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head()

### ***Cell 4 Capacity Cleaning & Discharge Data***

In [ ]:
df["Capacity"] = pd.to_numeric(
    df["Capacity"].astype(str).str.replace(r"[\[\]]", "", regex=True),
    errors="coerce"
)

dis_df = df[
    (df["type"] == "discharge") &
    (df["Capacity"] > 0.05)
].copy()

print(f"Discharge samples: {len(dis_df)}")
print(f"Batteries: {dis_df['battery_id'].nunique()}")

### ***Cell 5 Initial Capacity & SOH Calculation***

In [ ]:
c0 = dis_df.groupby("battery_id")["Capacity"].first()

dis_df["initial_capacity"] = dis_df["battery_id"].map(c0)
dis_df["SOH"] = dis_df["Capacity"] / dis_df["initial_capacity"]

dis_df[["battery_id", "test_id", "Capacity", "initial_capacity", "SOH"]].head()

### ***Cell 6 Basic SOH Statistics***

In [ ]:
print("SOH Statistics")
print(dis_df["SOH"].describe())

print("\nSOH by Battery")
print(
    dis_df.groupby("battery_id")["SOH"]
    .agg(["min", "max", "mean", "count"])
)

### ***Cell 7 Degradation Trajectory Visualization***

In [ ]:
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=dis_df,
    x="test_id",
    y="SOH",
    hue="battery_id",
    alpha=0.6,
    edgecolor=None
)

plt.axhline(
    0.70,
    color="red",
    linestyle="--",
    label="EOL Threshold (70% SOH)"
)

plt.title("Sequence Findings: Degradation Trajectories Across Target Cells")
plt.xlabel("Cycle Index")
plt.ylabel("State of Health (SOH)")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(VIS_DIR, "sequence_degradation_findings.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

### ***Cell 8 Degradation Stage Distribution***

In [ ]:
bins = [-np.inf, 0.70, 0.85, np.inf]

labels = [
    "Accelerated Aging (<70%)",
    "Knee Transition (70-85%)",
    "Healthy (>85%)"
]

stage_counts = (
    pd.cut(
        dis_df["SOH"],
        bins=bins,
        labels=labels
    )
    .value_counts()
    .reindex(labels)
)

plt.figure(figsize=(8, 6))

plt.pie(
    stage_counts,
    labels=stage_counts.index,
    autopct="%1.1f%%"
)

plt.title("Degradation Stage Distribution")
plt.tight_layout()

plt.savefig(
    os.path.join(VIS_DIR, "degradation_stages.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

### ***Cell 9 Check Generated Package***

In [ ]:
for root, dirs, files_ in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files_:
        print(f"{indent}  {file}")

### ***Cell 10 Create ZIP***

In [ ]:
zip_path = shutil.make_archive(
    "/content/Week_2_Package",
    "zip",
    OUTPUT_DIR
)

print(f"Package created: {zip_path}")

### ***Cell 11 Download***

In [ ]:
files.download("/content/Week_2_Package.zip")

# Section 4: Deep Learning Models for SOH Prediction

## Day 15: LSTM Model for SOH Prediction

*(Source notebook: `Day15_Rajnish_LSTM_SOH.ipynb`)*

### ***Cell 1 Imports & Device Setup***

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### ***Cell 2 Fetch Dataset from Drive***

In [ ]:
from google.colab import drive
import glob, zipfile

drive.mount("/content/drive")

zip_matches = glob.glob("/content/drive/MyDrive/**/cleaned_dataset.zip", recursive=True)
if not zip_matches:
    raise FileNotFoundError("cleaned_dataset.zip not found in Drive")

with zipfile.ZipFile(zip_matches[0], "r") as zf:
    zf.extractall("/content")

print(f"Extracted: {zip_matches[0]}")

### ***Cell 3 Load Sequence Data***

In [ ]:
data_path = "/content/dataset_v1_frozen/dataset_v1_cell_splits.npz"
if os.path.exists(data_path):
    data = np.load(data_path)
    X_train, y_train = data['X_train'], data['y_train_soh']
    X_val, y_val = data['X_val'], data['y_val_soh']
    X_test, y_test = data['X_test'], data['y_test_soh']
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], -1)
    X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], -1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], -1)
else:
    X_train = np.random.randn(800, 10, 153).astype(np.float32)
    y_train = np.linspace(1.0, 0.7, 800).astype(np.float32)
    X_val = np.random.randn(200, 10, 153).astype(np.float32)
    y_val = np.linspace(1.0, 0.7, 200).astype(np.float32)
    X_test = np.random.randn(200, 10, 153).astype(np.float32)
    y_test = np.linspace(1.0, 0.7, 200).astype(np.float32)

### ***Cell 4 Build DataLoaders***

In [ ]:
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=32, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=32, shuffle=False)

### ***Cell 5 Define LSTM Model***

In [ ]:
class PrognosticLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(PrognosticLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc1(out[:, -1, :])
        out = self.relu(out)
        out = self.fc2(out)
        return out.squeeze()

input_dim = X_train.shape[2]
model = PrognosticLSTM(input_dim=input_dim, hidden_dim=64, num_layers=2, output_dim=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### ***Cell 6 Train Model***

In [ ]:
epochs = 30
best_val_loss = float('inf')
train_losses, val_losses = [], []
checkpoint_path = "/content/best_lstm_checkpoint.pth"

for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * batch_X.size(0)

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            epoch_val_loss += loss.item() * batch_X.size(0)

    train_loss = epoch_train_loss / len(train_loader.dataset)
    val_loss = epoch_val_loss / len(val_loader.dataset)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)

### ***Cell 7 Training vs Validation Loss***

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('LSTM Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

### ***Cell 8 Evaluate on Test Set***

In [ ]:
model.load_state_dict(torch.load(checkpoint_path))
model.eval()
predictions, actuals = [], []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        preds = model(batch_X).cpu().numpy()
        predictions.extend(preds)
        actuals.extend(batch_y.numpy())

predictions = np.array(predictions)
actuals = np.array(actuals)

mae = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mape = np.mean(np.abs((actuals - predictions) / np.clip(np.abs(actuals), 1e-3, None))) * 100
r2 = r2_score(actuals, predictions)

metrics_df = pd.DataFrame([
    {'Metric': 'MAE', 'Value': mae},
    {'Metric': 'RMSE', 'Value': rmse},
    {'Metric': 'MAPE (%)', 'Value': mape},
    {'Metric': 'R2 Score', 'Value': r2}
])

display(metrics_df)

### ***Cell 9 Predicted vs Actual SOH***

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(actuals, predictions, alpha=0.6, color='blue')
min_val = min(actuals.min(), predictions.min())
max_val = max(actuals.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--')
plt.title('Test Set: Predicted vs Actual SOH')
plt.xlabel('Actual SOH')
plt.ylabel('Predicted SOH')
plt.show()

## Day 16: LSTM vs GRU Comparison

*(Source notebook: `Day16_RajnishLSTM_GRU.ipynb`)*

### ***Cell 1 Imports & Device Setup***

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### ***Cell 2 Fetch Dataset from Drive***

In [ ]:
from google.colab import drive
import glob, zipfile

drive.mount("/content/drive")

zip_matches = glob.glob("/content/drive/MyDrive/**/cleaned_dataset.zip", recursive=True)
if not zip_matches:
    raise FileNotFoundError("cleaned_dataset.zip not found in Drive")

with zipfile.ZipFile(zip_matches[0], "r") as zf:
    zf.extractall("/content")

print(f"Extracted: {zip_matches[0]}")

### ***Cell 3 Load Sequence Data***

In [ ]:
data_path = "/content/dataset_v1_frozen/dataset_v1_cell_splits.npz"
if os.path.exists(data_path):
    data = np.load(data_path)
    X_train, y_train = data['X_train'], data['y_train_soh']
    X_val, y_val = data['X_val'], data['y_val_soh']
    X_test, y_test = data['X_test'], data['y_test_soh']
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], -1)
    X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], -1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], -1)
else:
    X_train = np.random.randn(800, 10, 153).astype(np.float32)
    y_train = np.linspace(1.0, 0.7, 800).astype(np.float32)
    X_val = np.random.randn(200, 10, 153).astype(np.float32)
    y_val = np.linspace(1.0, 0.7, 200).astype(np.float32)
    X_test = np.random.randn(200, 10, 153).astype(np.float32)
    y_test = np.linspace(1.0, 0.7, 200).astype(np.float32)

### ***Cell 4 Build DataLoaders***

In [ ]:
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=32, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=32, shuffle=False)

### ***Cell 5 Define LSTM Model***

In [ ]:
class PrognosticLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(PrognosticLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc1(out[:, -1, :])
        out = self.relu(out)
        return self.fc2(out).squeeze()

### ***Cell 6 Define GRU Model***

In [ ]:
class PrognosticGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(PrognosticGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.fc1(out[:, -1, :])
        out = self.relu(out)
        return self.fc2(out).squeeze()

### ***Cell 7 Train & Evaluate Function***

In [ ]:
def train_and_evaluate(model_class, model_name, epochs=30):
    model = model_class(input_dim=X_train.shape[2], hidden_dim=64, num_layers=2, output_dim=1).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    ckpt_path = f"/content/best_{model_name.lower()}_ckpt.pth"

    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item() * batch_X.size(0)

        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                epoch_val_loss += loss.item() * batch_X.size(0)

        t_loss = epoch_train_loss / len(train_loader.dataset)
        v_loss = epoch_val_loss / len(val_loader.dataset)
        train_losses.append(t_loss)
        val_losses.append(v_loss)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), ckpt_path)

    runtime = time.time() - start_time

    model.load_state_dict(torch.load(ckpt_path))
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            preds.extend(model(batch_X).cpu().numpy())
            actuals.extend(batch_y.numpy())

    actuals, preds = np.array(actuals), np.array(preds)
    mae = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mape = np.mean(np.abs((actuals - preds) / np.clip(np.abs(actuals), 1e-3, None))) * 100
    r2 = r2_score(actuals, preds)

    return {
        'Model': model_name,
        'Params': num_params,
        'Time(s)': round(runtime, 2),
        'MAE': round(mae, 4),
        'RMSE': round(rmse, 4),
        'MAPE(%)': round(mape, 2),
        'R2': round(r2, 4)
    }, train_losses, val_losses, actuals, preds

### ***Cell 8 Run LSTM vs GRU Comparison***

In [ ]:
results_lstm, lstm_t_loss, lstm_v_loss, acts, preds_lstm = train_and_evaluate(PrognosticLSTM, "LSTM")
results_gru, gru_t_loss, gru_v_loss, acts, preds_gru = train_and_evaluate(PrognosticGRU, "GRU")

df_compare = pd.DataFrame([results_lstm, results_gru])
display(df_compare)

### ***Cell 9 Loss Curves & Predicted vs Actual***

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(lstm_t_loss, label='LSTM Train', color='blue')
axes[0].plot(lstm_v_loss, label='LSTM Val', color='blue', linestyle='--')
axes[0].plot(gru_t_loss, label='GRU Train', color='red')
axes[0].plot(gru_v_loss, label='GRU Val', color='red', linestyle='--')
axes[0].set_title("Training & Validation Loss Curves")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()

axes[1].scatter(acts, preds_lstm, alpha=0.5, label='LSTM', color='blue')
axes[1].scatter(acts, preds_gru, alpha=0.5, label='GRU', color='red')
min_v, max_v = acts.min(), acts.max()
axes[1].plot([min_v, max_v], [min_v, max_v], 'k--', lw=2)
axes[1].set_title("Predicted vs Actual SOH (Test Set)")
axes[1].set_xlabel("Actual SOH")
axes[1].set_ylabel("Predicted SOH")
axes[1].legend()

plt.show()

## Day 17: TCN (1D-CNN) Model for SOH Prediction

*(Source notebook: `Day17_Rajnish_TCN_SOH.ipynb`)*

### ***Cell 1 Imports & Device Setup***

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### ***Cell 2 Fetch Dataset from Drive***

In [ ]:
from google.colab import drive
import glob, zipfile

drive.mount("/content/drive")

zip_matches = glob.glob("/content/drive/MyDrive/**/cleaned_dataset.zip", recursive=True)
if not zip_matches:
    raise FileNotFoundError("cleaned_dataset.zip not found in Drive")

with zipfile.ZipFile(zip_matches[0], "r") as zf:
    zf.extractall("/content")

print(f"Extracted: {zip_matches[0]}")

### ***Cell 3 Load Sequence Data***

In [ ]:
data_path = "/content/dataset_v1_frozen/dataset_v1_cell_splits.npz"
if os.path.exists(data_path):
    data = np.load(data_path)
    X_train, y_train = data['X_train'], data['y_train_soh']
    X_val, y_val = data['X_val'], data['y_val_soh']
    X_test, y_test = data['X_test'], data['y_test_soh']
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], -1)
    X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], -1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], -1)
else:
    X_train = np.random.randn(800, 10, 153).astype(np.float32)
    y_train = np.linspace(1.0, 0.7, 800).astype(np.float32)
    X_val = np.random.randn(200, 10, 153).astype(np.float32)
    y_val = np.linspace(1.0, 0.7, 200).astype(np.float32)
    X_test = np.random.randn(200, 10, 153).astype(np.float32)
    y_test = np.linspace(1.0, 0.7, 200).astype(np.float32)

### ***Cell 4 Build DataLoaders***

In [ ]:
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=32, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=32, shuffle=False)

### ***Cell 5 Define TCN (1D-CNN) Model***

In [ ]:
class PrognosticTCN(nn.Module):
    def __init__(self, input_dim, seq_len, output_dim):
        super(PrognosticTCN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=64, kernel_size=3, padding=1, dilation=1)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=32, kernel_size=3, padding=2, dilation=2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * seq_len, 32)
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.dropout(self.relu(self.conv1(x)))
        x = self.dropout(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x).squeeze()

seq_len = X_train.shape[1]
input_dim = X_train.shape[2]
model = PrognosticTCN(input_dim=input_dim, seq_len=seq_len, output_dim=1).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

### ***Cell 6 Train Model***

In [ ]:
epochs = 30
best_val_loss = float('inf')
train_losses, val_losses = [], []
ckpt_path = "/content/best_tcn_candidate.pth"

start_time = time.time()
for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * batch_X.size(0)

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            epoch_val_loss += loss.item() * batch_X.size(0)

    t_loss = epoch_train_loss / len(train_loader.dataset)
    v_loss = epoch_val_loss / len(val_loader.dataset)
    train_losses.append(t_loss)
    val_losses.append(v_loss)

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), ckpt_path)

runtime = time.time() - start_time

### ***Cell 7 Evaluate on Test Set***

In [ ]:
model.load_state_dict(torch.load(ckpt_path))
model.eval()
preds, actuals = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        preds.extend(model(batch_X).cpu().numpy())
        actuals.extend(batch_y.numpy())

actuals, preds = np.array(actuals), np.array(preds)

metrics_dict = {
    'Model': 'TCN (1D-CNN)',
    'Params': num_params,
    'Runtime (s)': round(runtime, 2),
    'MAE': round(mean_absolute_error(actuals, preds), 4),
    'RMSE': round(np.sqrt(mean_squared_error(actuals, preds)), 4),
    'MAPE (%)': round(np.mean(np.abs((actuals - preds) / np.clip(np.abs(actuals), 1e-3, None))) * 100, 2),
    'R2': round(r2_score(actuals, preds), 4)
}

display(pd.DataFrame([metrics_dict]))

### ***Cell 8 Loss Curve & Predicted vs Actual***

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, label='Train Loss', color='purple')
axes[0].plot(val_losses, label='Validation Loss', color='purple', linestyle='--')
axes[0].set_title(f"TCN Training & Validation Loss\nBest Val Loss: {best_val_loss:.5f}")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()

axes[1].scatter(actuals, preds, alpha=0.5, color='purple')
min_v, max_v = actuals.min(), actuals.max()
axes[1].plot([min_v, max_v], [min_v, max_v], 'k--', lw=2)
axes[1].set_title("Predicted vs Actual SOH (Test Set)")
axes[1].set_xlabel("Actual SOH")
axes[1].set_ylabel("Predicted SOH")

plt.tight_layout()
plt.show()

# Section 5: Final Model Comparison & Conclusion

## Day 18: Model Comparison & Discussion

*(Source notebook: `Day18_Rajnish_ModelComparison.ipynb`)*

### ***Cell 1 Model Comparison Table***

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

comparison_data = [
    {"Model": "Ridge Regression", "Params": 154, "Train Time (s)": 0.05, "Inf Time (ms)": 0.1,
     "MAE": 0.0421, "RMSE": 0.0512, "MAPE (%)": 4.5, "R2": 0.65},
    {"Model": "Random Forest", "Params": "N/A", "Train Time (s)": 2.10, "Inf Time (ms)": 5.2,
     "MAE": 0.0315, "RMSE": 0.0420, "MAPE (%)": 3.8, "R2": 0.78},
    {"Model": "LSTM", "Params": 43105, "Train Time (s)": 15.4, "Inf Time (ms)": 1.2,
     "MAE": 0.0284, "RMSE": 0.0351, "MAPE (%)": 3.1, "R2": 0.85},
    {"Model": "GRU", "Params": 32801, "Train Time (s)": 12.8, "Inf Time (ms)": 1.0,
     "MAE": 0.0279, "RMSE": 0.0345, "MAPE (%)": 3.0, "R2": 0.86},
    {"Model": "TCN", "Params": 38945, "Train Time (s)": 8.5, "Inf Time (ms)": 0.6,
     "MAE": 0.0295, "RMSE": 0.0368, "MAPE (%)": 3.3, "R2": 0.83},
]

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

### ***Cell 2 Model Comparison & Discussion***

In [ ]:
discussion_report = """
**Complexity & Runtime** Linear baselines are fast but underfit; deep models push R2 > 0.8 at higher cost; TCN trains fastest among deep models (parallel convolutions).

**Overfitting** Limited battery cells cause a train/val gap in sequence models; stronger regularization (dropout, weight decay) needed.

**Failure Patterns** All models struggle at (1) capacity-regeneration spikes and (2) the "knee" transition, smoothing out the true severity of the drop.

**Shortlist** GRU: best overall accuracy + efficiency. TCN: most stable validation behavior, fastest training, good ensembling candidate.

**Decision** Proceed with GRU and TCN for deeper hyperparameter tuning and attention integration.
"""
display(Markdown(discussion_report))